In [1]:
import numpy as np
import pandas as pd
import math
import csv
import geopandas
import plotly.express as px
import requests
import time
import pulp

In [2]:
Locations = pd.DataFrame(columns=["Location","Latitude","Longitude"])
header = { "User-Agent": "TSP50Node"}
# alternate format:
# header = { "User-Agent": "AgentName (contact: emailID@domain.com)"}
ImportList = input("Load previously saved csv file location list?\nEnter'Y' for yes and 'N' for no\n")
if (ImportList == 'Y'):
    Locations = pd.read_csv(r"Locations.csv")

Entry = input("Enter place to visit:\nEnter 'stop' to stop\n")
while (Entry != 'stop'):
    time.sleep(1)
    if (Entry not in Locations["Location"]):

        query = {"q":Entry, "limit":1, "format":"jsonv2"}
        response = requests.get("https://nominatim.openstreetmap.org/search?",params=query,headers=header)
        # response = requests.get("https://photon.komoot.io/api/?",params=query)
        
        Locations.loc[len(Locations)] = [Entry, response.json()[0]['lat'], response.json()[0]['lon']]

    Entry = input("Enter place to visit:\n Enter 'stop' to stop\n")

In [3]:
Locations

,Location,Latitude,Longitude
0,"Pittsburgh, PA",40.440697,-80.002567
1,"Chicago, IL",41.875562,-87.624421
2,"Urbana, IL",40.111717,-88.207301
3,"Minneapolis, MN",44.977300,-93.265469
4,"Glacier National Park, MT",48.617486,-113.760885
5,"Seattle, WA",47.603832,-122.330062
6,"Boston, MA",42.358834,-71.057830
7,"San Diego, CA",32.715706,-117.163828
8,"Miami, FL",25.774157,-80.193597
9,"Albuquerque, NM",35.084103,-106.650985


In [4]:
RemoveLoc = input("Remove any location? 'Y' for yes and 'N' for no\n")
if RemoveLoc == 'Y':
    LocEntry = input("Enter Location: Enter 'stop' to stop\n")
    while (LocEntry != 'stop'):
        Locations = Locations[Locations["Location"] != LocEntry].reset_index(drop = True)
        LocEntry = input("Enter Location: Enter 'stop' to stop\n")        

In [5]:
Locations

,Location,Latitude,Longitude
0,"Pittsburgh, PA",40.440697,-80.002567
1,"Chicago, IL",41.875562,-87.624421
2,"Urbana, IL",40.111717,-88.207301
3,"Minneapolis, MN",44.977300,-93.265469
4,"Glacier National Park, MT",48.617486,-113.760885
5,"Seattle, WA",47.603832,-122.330062
6,"Boston, MA",42.358834,-71.057830
7,"San Diego, CA",32.715706,-117.163828
8,"Miami, FL",25.774157,-80.193597
9,"Albuquerque, NM",35.084103,-106.650985


In [6]:
if ImportList == 'Y':

    ImportedDistanceMatrix = np.genfromtxt("Distance.csv",delimiter=",")
    ImportedDurationMatrix = np.genfromtxt("Duration.csv",delimiter=",")

In [7]:
coordinates = ";".join(f"{lon},{lat}" for lat, lon in zip(Locations["Latitude"],Locations["Longitude"]))
url = f"https://router.project-osrm.org/table/v1/driving/{coordinates}"
response =requests.get(url,params={"annotations":"distance,duration"})

DistanceMatrix = np.array(response.json()["distances"])/1000 #kilometers
DurationMatrix = np.array(response.json()["durations"])/3600 #hours

In [8]:
Locations

,Location,Latitude,Longitude
0,"Pittsburgh, PA",40.440697,-80.002567
1,"Chicago, IL",41.875562,-87.624421
2,"Urbana, IL",40.111717,-88.207301
3,"Minneapolis, MN",44.977300,-93.265469
4,"Glacier National Park, MT",48.617486,-113.760885
5,"Seattle, WA",47.603832,-122.330062
6,"Boston, MA",42.358834,-71.057830
7,"San Diego, CA",32.715706,-117.163828
8,"Miami, FL",25.774157,-80.193597
9,"Albuquerque, NM",35.084103,-106.650985


In [9]:
SaveList = input("Save locations and distances?\nEnter 'Y' for yes 'N' for no\n")

if (SaveList == 'Y'):
    Locations.to_csv(r"Locations.csv", index = False)
    print("Location file saved")
    
    np.savetxt(r"Distance.csv", DistanceMatrix,
              delimiter=",")
    print("Distance file saved")

    np.savetxt(r"Duration.csv", DurationMatrix,
              delimiter=",")
    print("Duration file saved")

Location file saved
Distance file saved
Duration file saved


In [10]:
DistanceDict = pulp.makeDict([Locations["Location"],Locations["Location"]],DistanceMatrix,0)
DurationDict = pulp.makeDict([Locations["Location"],Locations["Location"]],DurationMatrix,0)
Arcs = [(i,j) for i in Locations["Location"] for j in Locations["Location"]]

model = pulp.LpProblem("RouteTSP", pulp.LpMinimize)

x = model.add_variable_dicts("X", (Locations["Location"],Locations["Location"]), cat = "Binary")
# z = model.add_variable("Z")

# model += (z,"Route Distance")

model += (pulp.lpSum([x[i][j] * DistanceDict[i][j] for (i,j) in Arcs]),"Route Distance")

# model += (z == pulp.lpSum([x[i][j] * DistanceDict[i][j] for (i,j) in Arcs]), "Objective Definition")
for i in Locations["Location"]:
    model += (pulp.lpSum(x[i][j] for j in Locations["Location"][Locations["Location"] != i]) == 1, f"Outflow from {i}")
    model += (pulp.lpSum(x[j][i] for j in Locations["Location"][Locations["Location"] != i]) == 1, f"Inflow into {i}")

startTime = time.perf_counter()
model.solve(pulp.getSolver('HiGHS',gapRel=0.0))

OptimalDistance = round(model.objective.value(),3)
print(f"The route is {OptimalDistance} kilometers.")

# model

The route is 22287.467 kilometers.


In [11]:
def Cycle(start, graph):
    VisitOrder = [start]
    current = start
    
    while (graph[current] != start):
        VisitOrder.append(graph[current])
        current = graph[current]
    VisitOrder.append(start)
    return VisitOrder    

In [12]:
def CycleConnect(cycles):
    n1 = len(cycles[0]) - 1
    n2 = len(cycles[1]) - 1
    breakCost = [[0]*n2 for _ in range(n1)]
    minCost = np.inf
    breakIndex = [0,0]
    for i in range(n1):
        for j in range(n2):
            # print(i,j,len(cycles[0]),len(cycles[1]),len(breakCost[0]),len(breakCost))
            # print(breakCost)
            
            breakCost[i][j] = (DistanceDict[cycles[0][i]][cycles[1][j+1]] + DistanceDict[cycles[1][j]][cycles[0][i+1]] - DistanceDict[cycles[0][i]][cycles[0][i+1]] - DistanceDict[cycles[1][j]][cycles[1][j+1]])
            if breakCost[i][j] < minCost:
                minCost = breakCost[i][j]
                breakIndex = [i,j]
    Cycle = []
    for i in range(breakIndex[0]+1):
        Cycle.append(cycles[0][i])
    for i in range(breakIndex[1]+1,len(cycles[1])):
        Cycle.append(cycles[1][i])
    for i in range(1,breakIndex[1]+1):
        Cycle.append(cycles[1][i])
    for i in range(breakIndex[0]+1,len(cycles[0])):
        Cycle.append(cycles[0][i])

    return Cycle, minCost

In [13]:
def printResult(VisitOrder, startTime):
    RouteString = "The route is:"
    for i in range(len(VisitOrder)-1):
        RouteString += VisitOrder[i]
        RouteString += "->"
    RouteString = RouteString[:-2]
    print(RouteString)
    endTime = time.perf_counter()
    runTime = endTime - startTime
    print(f"\nThe solver took {runTime/60} minutes.")

In [14]:
cycles = [[1]]
it = 1
epsilon = 0.005 # can adjust this for solver performance. High epsilon -> faster runtime, low epsilon -> better quality solution
maxTime = 60 # max run time in minutes. Adjustable
minRouteCost = np.inf
while len(cycles[0]) < len(Locations):
    graph = {}
    cycles = []
    
    for i in Locations["Location"]:
        for j in Locations["Location"][Locations["Location"] != i]:
            if round(x[i][j].value()) == 1:
                graph[i] = j
    
    Visited = {}
    for i in Locations["Location"]:
        Visited[i] = False
        
    for i in Locations["Location"]:
        if not Visited[i]:
            VisitOrder = Cycle(i, graph)
            cycles.append(VisitOrder)
            model += (pulp.lpSum(x[VisitOrder[i]][VisitOrder[i+1]] for i in range(len(VisitOrder)-1)) <= (len(VisitOrder) - 2), 
                      f"Subtour elimination number {len(cycles)} from iteration {it}")
            for j in VisitOrder:
                Visited[j] = True
                
    if len(cycles[0]) < len(Locations):
        print(f"Iteration {it}: {len(cycles)} subtours found. Current objective: {OptimalDistance}")
        
        if len(cycles) == 2:
            # Construct a tour by stitching 2 subtours for a heuristic upper bound which can be used if it becomes sufficiently useful
            VisitOrder, AdjustedCost = CycleConnect(cycles)
            elapsedTime = time.perf_counter() - startTime
        
            if (OptimalDistance + AdjustedCost) < minRouteCost:
                minRouteCost = OptimalDistance + AdjustedCost
                currentBestHeuristicRoute = VisitOrder
                print(f"Heuristic tour obtained by stitching one tour from the two subtours with cost: {OptimalDistance + AdjustedCost}")
        
            if elapsedTime >= 60*maxTime and minRouteCost < np.inf:
                print(f"\n\n\n============\nTerminated after running for {maxTime} minutes\n============")
                print(f"Best objective: {minRouteCost}; Best bound: {OptimalDistance}")
                OptimalDistance = minRouteCost
                VisitOrder = currentBestHeuristicRoute
                printResult(VisitOrder, startTime)
                break
        
            if minRouteCost <= (1+epsilon) * OptimalDistance:
                OptimalDistance = minRouteCost
                VisitOrder = currentBestHeuristicRoute
                print(f"\n\n\n============\nTerminated with <={(1+epsilon)} times the optimal solution\n============")
                print(f"Best objective: {OptimalDistance}")
                printResult(VisitOrder, startTime)
                break  
            
        model.solve()
        OptimalDistance = round(model.objective.value(),3)
        it += 1                                           
    
    else:
        print("\n\n\n============\nFound optimal solution!\n============")
        print(f"Iteration {it}: {len(cycles)} subtours found. Optimal objective: {OptimalDistance}")
        it += 1
        printResult(VisitOrder, startTime)

Iteration 1: 17 subtours found. Current objective: 22287.467
Iteration 2: 9 subtours found. Current objective: 24544.698
Iteration 3: 9 subtours found. Current objective: 24791.796
Iteration 4: 7 subtours found. Current objective: 25304.417
Iteration 5: 5 subtours found. Current objective: 25443.89
Iteration 6: 5 subtours found. Current objective: 25605.198
Iteration 7: 3 subtours found. Current objective: 25617.796
Iteration 8: 2 subtours found. Current objective: 25619.951
Heuristic tour obtained by stitching one tour from the two subtours with cost: 27206.7647
Iteration 9: 4 subtours found. Current objective: 25667.324
Iteration 10: 2 subtours found. Current objective: 25677.593
Heuristic tour obtained by stitching one tour from the two subtours with cost: 26901.414500000003
Iteration 11: 5 subtours found. Current objective: 25691.978
Iteration 12: 4 subtours found. Current objective: 25705.984
Iteration 13: 2 subtours found. Current objective: 25819.994
Iteration 14: 3 subtours fou

In [15]:
print("Distance in miles: ",OptimalDistance/1.6)

Distance in miles:  16414.804375


In [16]:
TotalDuration = 0
iIndex = 0

for i in Locations["Location"]:
    jIndex = 0
    for j in Locations["Location"]:
        # print(iIndex, i, jIndex, j)
        if i != j:
            if round(x[i][j].value()) == 1:
                TotalDuration += DurationMatrix[iIndex][jIndex]
        
        jIndex += 1
    iIndex += 1

In [17]:
Durations = [0]
Distances = [0]
for i in range(1,len(VisitOrder)):
    Durations.append(DurationDict[VisitOrder[i-1]][VisitOrder[i]])
    Distances.append(DistanceDict[VisitOrder[i-1]][VisitOrder[i]])

OrderedLocations = pd.DataFrame(columns=["Location","Latitude","Longitude"])

for i in range(len(VisitOrder)):
    OrderedLocations = pd.concat([OrderedLocations, Locations.loc[Locations["Location"] == VisitOrder[i]]] , ignore_index = True)

OrderedLocations.insert(3, "Distance (in kms)", Distances)
OrderedLocations.insert(4, "Duration (in hrs)", Durations)
OrderedLocations

,Location,Latitude,Longitude,Distance (in kms),Duration (in hrs)
0,"Pittsburgh, PA",40.440697,-80.002567,0.0000,0.000000
1,"Washington DC, DC",38.895098,-77.036385,390.5713,4.944639
2,"Philadelphia, PA",39.952724,-75.163526,221.6704,3.059278
3,"New York, NY",40.712728,-74.006015,151.2784,2.085083
4,"Boston, MA",42.358834,-71.05783,343.6284,4.677556
5,"Acadia National Park, ME",44.356311,-68.215479,429.2383,5.773417
6,"Quebec City, QC",46.813743,-71.208406,457.0669,6.389222
7,"Montreal, QC",45.503182,-73.569806,251.6879,3.440833
8,"Toronto, ON",43.653482,-79.383935,542.0825,6.529444
9,"Niagara Falls, ON",43.10656,-79.063904,130.3699,1.631194


In [18]:
OrderedLocations.to_csv("OrderedLocations.csv")

In [19]:
fig = px.line_geo(OrderedLocations,lat="Latitude",lon="Longitude",hover_name="Location",markers=True)
fig.update_geos(fitbounds="locations")
fig.update_layout(geo_scope="north america",height=500,showlegend=True)
fig.show()

In [20]:
fig.write_html("trip.html")

In [21]:
DurationHours = int(TotalDuration)
DurationMinutes = math.ceil((TotalDuration % 1) * 60)

if (DurationMinutes == 60):
    DurationHours += 1
    DurationMinutes = 0

TripDays = math.ceil(TotalDuration / 6)

print("The trip takes",DurationHours,"hours and",DurationMinutes,"minutes.")
print("At 6 hours per day, it takes",TripDays,"days to finish the trip.")

The trip takes 336 hours and 42 minutes.
At 6 hours per day, it takes 57 days to finish the trip.
